In [4]:
import pandas as pd
from pandas.api.types import is_numeric_dtype, is_string_dtype
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score,mean_squared_error, r2_score, mean_absolute_error, f1_score, roc_auc_score
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit, train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [9]:
# Movies

data_dir = "../data/raw/"

movies = pd.read_csv(os.path.join(data_dir, "short_movie.csv")).drop_duplicates(subset=["movie_id"])
movies = movies.set_index("movie_id").drop(["Unnamed: 0"], axis=1)
num_cols = movies.select_dtypes(include="number").columns
movies[num_cols] = movies[num_cols].astype(float)
movies.drop(columns=["original_title"], inplace=True)

movies.rename(
    columns={
        "title": "Title",
        "imdb_id": "IMDB ID",
        "release_date": "Release Date",
        "language": "Language",
        "runtime": "Length (min)",
        "budget": "Budget (USD)",
        "rating": "Rating",
        "revenue": "Revenue (USD)",
        "popularity": "Popularity",
        "vote_average": "Rating",
        "vote_count": "Rating Count",
        "cast_size": "Cast Size",
        "crew_size": "Crew Size"
    },
    inplace=True
)


# Cast and Crew

casting = pd.DataFrame()
for dir in os.listdir(data_dir):
    if dir.startswith("person_") and not dir.startswith("person_movie_history"):
        temp_person = pd.read_csv(os.path.join(data_dir, dir))
        casting = pd.concat([casting, temp_person])


casting = casting.rename(columns={"Unnamed: 0": "cast_id"}).set_index("cast_id")
casting.rename(columns={
    "id": "pid",
    "movie_id": "Movie ID",
    "name": "Name",
    "gender": "Gender",
    "popularity": "Popularity",
    "job": "Job"
}, inplace=True)
casting.drop(columns=["original_name"], inplace=True)

In [10]:
metadata_movies = movies.iloc[:, ~movies.columns.str.contains(r"^genre")].copy()

genre_movies = movies.loc[:, movies.columns.str.contains(r"^genre_")].copy()

display(metadata_movies.head(5))
print("Movies dataset shape:", movies.shape)
print("\n\n")

display(genre_movies.head(4))
print("Casting dataset shape:", casting.shape)

,IMDB ID,Title,Language,Popularity,Rating Count,Rating,Release Date,Length (min),Budget (USD),Revenue (USD),Cast Size,Crew Size
movie_id,,,,,,,,,,,,
10898,tt0240684,The Little Mermaid II: Return to the Sea,en,1.3106,1766.0,6.401,2000-01-23,75.0,0.0,0.0,20.0,20.0
49948,tt0120910,Fantasia 2000,en,4.2748,1354.0,7.000,2000-01-01,74.0,80000000.0,60655420.0,15.0,70.0
10471,tt0195945,Next Friday,en,2.4630,631.0,6.469,2000-01-12,98.0,11000000.0,59827328.0,27.0,121.0
2360,tt0195234,Saving Grace,en,3.5571,431.0,6.500,2000-01-24,93.0,10000000.0,26330482.0,26.0,28.0
10384,tt0134983,Supernova,en,1.4180,425.0,4.900,2000-01-14,91.0,90000000.0,14828081.0,12.0,170.0


Movies dataset shape: (18521, 32)





,genre_Action,genre_Adventure,genre_Animation,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Family,genre_Fantasy,genre_History,genre_Horror,genre_Music,genre_Mystery,genre_Romance,genre_Science Fiction,genre_TV Movie,genre_Thriller,genre_War,genre_Western
movie_id,,,,,,,,,,,,,,,,,,,
10898,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49948,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10471,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2360,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Casting dataset shape: (572733, 6)


In [12]:
# NaN Value and Zero Values

summary_movies = pd.DataFrame({
    "Non-Missing Values": metadata_movies.notna().sum(),
    "Missing Values": metadata_movies.isna().sum(),
    "Zero Values": (metadata_movies == 0).sum()
})

# Change from Zero Values into np.NaN for some column
non_zero_cols = ["Length (min)", "Budget (USD)", "Revenue (USD)", "Popularity"]
metadata_movies.loc[:, non_zero_cols] = metadata_movies.loc[:, non_zero_cols].replace({0: np.nan})


# NaN Value and Zero Values

nan_values = casting.isna().sum()
zero_values = []

for col in nan_values.index:
  if is_numeric_dtype(casting[col]):
    zero_values.append(sum(casting[col] == 0))

  elif is_string_dtype(casting[col]):
    zero_values.append(sum(casting[col] == ""))


summary_casting = pd.DataFrame({
    "Non-Missing Values": casting.notna().sum(),
    "Missing Values": casting.isna().sum(),
    "Zero Values": zero_values
})

summary = pd.concat([
    summary_movies,
    summary_casting
])
summary = summary.T
display(summary)

casting["Popularity"] = casting["Popularity"].replace({0: np.nan})
casting["Gender"] = casting["Gender"].replace({2: -1, 3: 0})

,IMDB ID,Title,Language,Popularity,Rating Count,Rating,Release Date,Length (min),Budget (USD),Revenue (USD),Cast Size,Crew Size,pid,Movie ID,Name,Gender,Popularity,Job
Non-Missing Values,18456,18520,18521,18520,18521,18521,18521,18426,5710,6405,18521,18521,572733,572733,572733,572733,572540,572733
Missing Values,65,1,0,1,0,0,0,95,12811,12116,0,0,0,0,0,0,193,0
Zero Values,0,0,0,0,0,0,0,0,0,0,161,28,0,0,0,56549,0,0


In [13]:
metadata_movies.loc[:, "Profit (USD)"] = metadata_movies["Revenue (USD)"] - metadata_movies["Budget (USD)"]
metadata_movies.loc[:, "ROI"] = np.where(metadata_movies["Budget (USD)"] > 0, metadata_movies["Revenue (USD)"] / metadata_movies["Budget (USD)"], np.nan)

# 90th percentile of revenue by year
metadata_movies["yearly_revenue_p90"] = (
    metadata_movies
    .groupby("Release Date")["Revenue (USD)"]
    .transform(lambda x: x.quantile(0.90))
)

# Define blockbuster relative to year
metadata_movies["Blockbuster"] = (
    metadata_movies["Revenue (USD)"] >= metadata_movies["yearly_revenue_p90"]
)
metadata_movies.drop(columns=["yearly_revenue_p90"], inplace=True)

# Filtering ROI due to unexpectedly high value
threshold = metadata_movies["ROI"].quantile(0.99)
filtered = (metadata_movies["ROI"].isna()) | (metadata_movies["ROI"] < threshold)

print("Unfiltered ROI Movie Observation:", len(metadata_movies))

metadata_movies = metadata_movies[filtered]
print("Filtered ROI Movie Observation:", len(metadata_movies))

Unfiltered ROI Movie Observation: 18521
Filtered ROI Movie Observation: 18479


In [14]:
# Filtering Language
threshold = 200 # Hard-coded based on manual observation

series = pd.Series(metadata_movies["Language"].value_counts())
series[series > threshold]
valid_lang = series[series > threshold].index

print("Total Language:", len(metadata_movies["Language"].unique()), ", Total Observations:", len(metadata_movies))

metadata_movies = metadata_movies[metadata_movies["Language"].isin(valid_lang)]

print("Valid Language:", len(valid_lang), ", Total Observations:", len(metadata_movies))

Total Language: 74 , Total Observations: 18479
Valid Language: 11 , Total Observations: 16772


In [15]:
# Using Bayesian Weighted Average Rating

# Quantile for count of pseudo_votes
pseudo_votes = metadata_movies["Rating Count"].quantile(0.25)


r = metadata_movies["Rating"]
avg_r = metadata_movies["Rating"].mean()
v = metadata_movies["Rating Count"]

metadata_movies["Weighted Rating"] = (r  * v / (v + pseudo_votes) + 
                             avg_r* pseudo_votes / (v + pseudo_votes))
summary = metadata_movies["Weighted Rating"].describe()
print("Weighted Rating Summary:")

result = pd.DataFrame([{
    "count": summary["count"],
    "mean": round(summary["mean"], 2),
    "min": round(summary["min"], 2),
    "max": round(summary["max"], 2)
}])
display(result.style.hide(axis="index").format("{:.2f}"))

Weighted Rating Summary:


count,mean,min,max
16772.00,6.22,3.00,8.69


In [16]:
movies = metadata_movies.merge(genre_movies, left_index=True, right_index=True)
display(movies.head(5))

,IMDB ID,Title,Language,Popularity,Rating Count,Rating,Release Date,Length (min),Budget (USD),Revenue (USD),...,genre_History,genre_Horror,genre_Music,genre_Mystery,genre_Romance,genre_Science Fiction,genre_TV Movie,genre_Thriller,genre_War,genre_Western
movie_id,,,,,,,,,,,,,,,,,,,,,
10898,tt0240684,The Little Mermaid II: Return to the Sea,en,1.3106,1766.0,6.401,2000-01-23,75.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49948,tt0120910,Fantasia 2000,en,4.2748,1354.0,7.000,2000-01-01,74.0,80000000.0,60655420.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10471,tt0195945,Next Friday,en,2.4630,631.0,6.469,2000-01-12,98.0,11000000.0,59827328.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2360,tt0195234,Saving Grace,en,3.5571,431.0,6.500,2000-01-24,93.0,10000000.0,26330482.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10384,tt0134983,Supernova,en,1.4180,425.0,4.900,2000-01-14,91.0,90000000.0,14828081.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [17]:
display(casting.head(3))
print("Casting dataset shape:", casting.shape)
print("Total Unique Person:", len(casting["pid"].unique()))

,pid,Movie ID,Name,Gender,Popularity,Job
cast_id,,,,,,
0,63978,10898,Jodi Benson,1,0.8555,Acting
1,67392,10898,Samuel E. Wright,-1,0.4486,Acting
2,15762,10898,Tara Strong,1,2.0106,Acting


Casting dataset shape: (572733, 6)
Total Unique Person: 132501


In [18]:
# NaN Value and Zero Values

nan_values = casting.isna().sum()
zero_values = []

for col in nan_values.index:
  if is_numeric_dtype(casting[col]):
    zero_values.append(sum(casting[col] == 0))

  elif is_string_dtype(casting[col]):
    zero_values.append(sum(casting[col] == ""))


summary = pd.DataFrame({
    "Missing  Values": casting.isna().sum(),
    "Zero Values": zero_values
})
summary = summary.T
display(summary)

casting["Popularity"] = casting["Popularity"].replace({0: np.nan})
casting["Gender"] = casting["Gender"].replace({2: -1, 3: 0})

,pid,Movie ID,Name,Gender,Popularity,Job
Missing Values,0,0,0,0,193,0
Zero Values,0,0,0,56549,0,0


In [21]:
movies.to_csv("../data/processed/movies.csv")
casting.to_csv("../data/processed/casting.csv")